In [1]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import subprocess
import warnings

warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package])

try:
    from astroquery.heasarc import Heasarc
    import astropy
    import scipy
    import matplotlib
except ImportError:
    print("[SYSTEM] Desplegando Astroquery...")
    install("astroquery")
    install("astropy")
    install("scipy")
    install("matplotlib")
    from astroquery.heasarc import Heasarc

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit

print("[SYSTEM] Iniciando Protocolo ASTRO-MINER (HEASARC Object Query)...")

# --- BLOQUE 1: EXTRACCIÓN SOBERANA ---
heasarc = Heasarc()
TARGET_TABLE = 'fermilgrb'

# Definimos las columnas EXACTAS que necesitamos.
REQUIRED_FIELDS = 'NAME,REDSHIFT,EXT_EMISSION_MAX_ENE,EXT_EMISSION_MAX_ENE_T'

print(f"[QUERY] Descargando tabla '{TARGET_TABLE}' usando query_object('%')...")

try:
    # CORRECCIÓN: Usamos query_object con comodín '%' para bajar toda la tabla de la misión especificada.
    # No usamos query_mission_list porque no acepta argumentos de filtrado en esta versión.
    table = heasarc.query_object(object_name='%', mission=TARGET_TABLE,
                                 fields=REQUIRED_FIELDS, result_max=3000)

    df = table.to_pandas()
    # Normalizar a mayúsculas
    df.columns = [c.upper() for c in df.columns]
    print(f"[SUCCESS] Tabla recuperada. Registros: {len(df)}")
    # print(f"[INFO] Columnas recibidas: {df.columns.tolist()}")

except Exception as e:
    print(f"[CRITICAL] Error de conexión HEASARC: {e}")
    # Si falla con '%', intentamos sin object_name si la librería lo permite, o con un nombre genérico
    sys.exit()

# --- BLOQUE 2: MAPEO Y LIMPIEZA ---
try:
    raw_z = df['REDSHIFT']
    raw_e = df['EXT_EMISSION_MAX_ENE']     # MeV
    raw_t = df['EXT_EMISSION_MAX_ENE_T']   # s
    raw_n = df['NAME']
except KeyError as e:
    print(f"[ERROR] HEASARC no devolvió la columna solicitada: {e}")
    # Fallback de emergencia por si los nombres cambiaron ligeramente
    print(f"[DEBUG] Columnas disponibles: {list(df.columns)}")
    sys.exit()

# Convertir a GeV
raw_e_gev = raw_e / 1000.0

# Filtros Físicos
# 1. Redshift definido
# 2. Energía > 1 GeV (Para tener señal LIV)
# 3. Tiempo definido
mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 1.0) & (np.isfinite(raw_t))

clean_z = raw_z[mask].values
clean_t = raw_t[mask].values
clean_e = raw_e_gev[mask].values
clean_names = raw_n[mask].values

print(f"[DATA] Eventos LIV Válidos (z>0, E>1GeV): {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Pocos datos extremos. Bajando umbral a 100 MeV...")
    mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 0.1) & (np.isfinite(raw_t))
    clean_z = raw_z[mask].values
    clean_t = raw_t[mask].values
    clean_e = raw_e_gev[mask].values
    clean_names = raw_n[mask].values
    print(f"[DATA] Eventos recuperados: {len(clean_z)}")

# --- BLOQUE 3: CÁLCULO K-FACTOR ---
print("[PHYS] Calculando integrales cosmológicas...")
def get_k(z):
    z_grid = np.linspace(0, z, 50)
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

if len(clean_z) > 0:
    K_factors = np.array([get_k(z) for z in clean_z])
    # Observable LIV: Retraso dividido por energía
    # tau = dt / E
    lags = clean_t / clean_e
else:
    K_factors = []
    lags = []

# --- BLOQUE 4: REGRESIÓN Y VISUALIZACIÓN ---
def model(k, m, c):
    return m * k + c

if len(K_factors) > 2:
    # Ajuste robusto ignorando outliers masivos
    try:
        # Filtro sigma clipping simple para quitar outliers astrofísicos locos
        q1 = np.percentile(lags, 25)
        q3 = np.percentile(lags, 75)
        iqr = q3 - q1
        filter_mask = (lags >= q1 - 1.5*iqr) & (lags <= q3 + 1.5*iqr)

        popt, pcov = curve_fit(model, K_factors[filter_mask], lags[filter_mask])
        m_fit = popt[0]
        m_err = np.sqrt(np.diag(pcov))[0]
    except:
        m_fit, m_err = 0, 0
else:
    m_fit, m_err = 0, 0

plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

if len(K_factors) > 0:
    # Plot datos
    sc = plt.scatter(K_factors, lags, c=clean_e, cmap='plasma', s=100, alpha=0.9, label='Datos Fermi-LAT')
    plt.colorbar(sc, label='Energía (GeV)')

    # Plot ajuste
    if m_fit != 0:
        x_plot = np.linspace(0, max(K_factors)*1.1, 50)
        plt.plot(x_plot, model(x_plot, m_fit, popt[1]), color='#00ffcc',
                 linestyle='--', lw=2, label=f'Ajuste LIV (m={m_fit:.2e})')

    plt.axhline(0, color='white', linestyle=':', label='Einstein (m=0)')

    # Etiquetas inteligentes
    for i, name in enumerate(clean_names):
        # Etiquetar solo si tiene alto retraso o alta energía
        if clean_e[i] > 5.0 or abs(lags[i]) > 5:
            try:
                n_str = name.decode('utf-8') if isinstance(name, bytes) else str(name)
            except: n_str = str(name)
            plt.annotate(f"{n_str}\n{clean_e[i]:.1f}GeV",
                         (K_factors[i], lags[i]), color='yellow', fontsize=8,
                         xytext=(5, 5), textcoords='offset points')

plt.xlabel(r'Factor de Distancia Cosmológica $K(z)$')
plt.ylabel(r'Retraso Normalizado $\tau_n = \Delta t / E$ (s/GeV)')
plt.title('Cazando Violación de Lorentz (Final v2)\nDatos: HEASARC fermilgrb [EXT_EMISSION]', color='white')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[RESULT] Pendiente LIV: m = {m_fit:.4e} +/- {m_err:.4e} s/GeV")

[SYSTEM] Desplegando Astroquery...
[SYSTEM] Iniciando Protocolo ASTRO-MINER (HEASARC Object Query)...
[QUERY] Descargando tabla 'fermilgrb' usando query_object('%')...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[CRITICAL] Error de conexión HEASARC: Unable to find coordinates for name '%' using https://cds.unistra.fr/cgi-bin/nph-sesame/SNV?%25
Traceback (most recent call last):
  File "/tmp/ipython-input-304279557.py", line 43, in <cell line: 0>
    table = heasarc.query_object(object_name='%', mission=TARGET_TABLE,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/utils/decorators.py", line 143, in deprecated_func
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astroquery/heasarc/core.py", line 463, in query_object
    pos = coordinates.SkyCoord.from_name(object_name)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/coordinates/sky_coordinate.py", line 1905, in from_name
    icrs_coord = get_icrs_coordinates(name, parse, cache=cache)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

TypeError: object of type 'NoneType' has no len()

In [2]:
# --- BLOQUE 0: INFRAESTRUCTURA ---
import sys
import subprocess
import warnings

warnings.filterwarnings('ignore')

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package])

try:
    from astroquery.heasarc import Heasarc
    import astropy
    import scipy
    import matplotlib
except ImportError:
    print("[SYSTEM] Desplegando Astroquery...")
    install("astroquery")
    install("astropy")
    install("scipy")
    install("matplotlib")
    from astroquery.heasarc import Heasarc

import numpy as np
import matplotlib.pyplot as plt
from astropy.cosmology import Planck18 as cosmo
from scipy.optimize import curve_fit

print("[SYSTEM] Iniciando Protocolo FINAL (Explicit Fields Request)...")

# --- BLOQUE 1: EXTRACCIÓN QUIRÚRGICA ---
heasarc = Heasarc()
TARGET_TABLE = 'fermilgrb'

# ESTA ES LA CLAVE: Pedimos explícitamente las columnas que TÚ encontraste.
# No dejamos que la API decida.
# Nota: HEASARC suele aceptar mayúsculas/minúsculas indistintamente, pero usamos las de tu lista.
REQ_FIELDS = 'NAME,REDSHIFT,HIGHEST_ENERGY,HIGHEST_ENERGY_TIME'

print(f"[QUERY] Solicitando columnas blindadas: {REQ_FIELDS}")

try:
    # CORRECCIÓN: Usamos query_mission para obtener toda la tabla de la misión especificada.
    # query_object requiere un nombre de objeto válido para resolución de coordenadas.
    table = heasarc.query_mission(mission_name=TARGET_TABLE,
                                  fields=REQ_FIELDS, result_max=3000)

    df = table.to_pandas()
    # Normalizar a mayúsculas
    df.columns = [c.upper() for c in df.columns]
    print(f"[SUCCESS] Tabla recuperada. Registros: {len(df)}")
    print(f"[INFO] Columnas recibidas: {df.columns.tolist()}")

except Exception as e:
    print(f"[CRITICAL] Error de conexión HEASARC: {e}")
    sys.exit()

# --- BLOQUE 2: MAPEO Y LIMPIEZA ---
try:
    raw_z = df['REDSHIFT']
    # En tu lista aparece como HIGHEST_ENERGY (MeV)
    raw_e = df['HIGHEST_ENERGY']
    # En tu lista aparece como HIGHEST_ENERGY_TIME (s)
    raw_t = df['HIGHEST_ENERGY_TIME']
    raw_n = df['NAME']
except KeyError as e:
    print(f"[ERROR] HEASARC no devolvió la columna solicitada: {e}")
    print(f"[DEBUG] Columnas disponibles: {list(df.columns)}")
    sys.exit()

# Convertir a GeV (asumiendo que viene en MeV según tu texto "100 MeV-100 GeV")
raw_e_gev = raw_e / 1000.0

# Filtros Físicos
# 1. Redshift definido
# 2. Energía > 1 GeV (Para tener señal LIV y limpiar ruido)
# 3. Tiempo definido
mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 1.0) & (np.isfinite(raw_t))

clean_z = raw_z[mask].values
clean_t = raw_t[mask].values
clean_e = raw_e_gev[mask].values
clean_names = raw_n[mask].values

print(f"[DATA] Eventos LIV Válidos (z>0, E>1GeV): {len(clean_z)}")

if len(clean_z) < 3:
    print("[WARN] Pocos datos extremos. Bajando umbral a 100 MeV...")
    mask = (raw_z > 0) & (np.isfinite(raw_z)) & (raw_e_gev > 0.1) & (np.isfinite(raw_t))
    clean_z = raw_z[mask].values
    clean_t = raw_t[mask].values
    clean_e = raw_e_gev[mask].values
    clean_names = raw_n[mask].values
    print(f"[DATA] Eventos recuperados: {len(clean_z)}")

# --- BLOQUE 3: CÁLCULO K-FACTOR ---
print("[PHYS] Calculando integrales cosmológicas...")
def get_k(z):
    z_grid = np.linspace(0, z, 50)
    Ez = np.sqrt(cosmo.Om0*(1+z_grid)**3 + cosmo.Ode0)
    integrand = (1+z_grid) / Ez
    return np.trapz(integrand, z_grid)

if len(clean_z) > 0:
    K_factors = np.array([get_k(z) for z in clean_z])
    # Observable LIV: Retraso dividido por energía
    # tau = dt / E
    lags = clean_t / clean_e
else:
    K_factors = []
    lags = []

# --- BLOQUE 4: REGRESIÓN Y VISUALIZACIÓN ---
def model(k, m, c):
    return m * k + c

if len(K_factors) > 2:
    # Ajuste robusto ignorando outliers masivos
    try:
        # Filtro sigma clipping simple para quitar outliers astrofísicos locos
        q1 = np.percentile(lags, 25)
        q3 = np.percentile(lags, 75)
        iqr = q3 - q1
        filter_mask = (lags >= q1 - 1.5*iqr) & (lags <= q3 + 1.5*iqr)

        popt, pcov = curve_fit(model, K_factors[filter_mask], lags[filter_mask])
        m_fit = popt[0]
        m_err = np.sqrt(np.diag(pcov))[0]
    except:
        m_fit, m_err = 0, 0
else:
    m_fit, m_err = 0, 0

plt.figure(figsize=(12, 7))
plt.style.use('dark_background')

if len(K_factors) > 0:
    # Plot datos
    sc = plt.scatter(K_factors, lags, c=clean_e, cmap='plasma', s=100, alpha=0.9, label='Datos Fermi-LAT')
    plt.colorbar(sc, label='Energía (GeV)')

    # Plot ajuste
    if m_fit != 0:
        x_plot = np.linspace(0, max(K_factors)*1.1, 50)
        plt.plot(x_plot, model(x_plot, m_fit, popt[1]), color='#00ffcc',
                 linestyle='--', lw=2, label=f'Ajuste LIV (m={m_fit:.2e})')

    plt.axhline(0, color='white', linestyle=':', label='Einstein (m=0)')

    # Etiquetas inteligentes
    for i, name in enumerate(clean_names):
        # Etiquetar solo si tiene alto retraso o alta energía
        if clean_e[i] > 5.0 or abs(lags[i]) > 5:
            try:
                n_str = name.decode('utf-8') if isinstance(name, bytes) else str(name)
            except: n_str = str(name)
            plt.annotate(f"{n_str}\n{clean_e[i]:.1f}GeV",
                         (K_factors[i], lags[i]), color='yellow', fontsize=8,
                         xytext=(5, 5), textcoords='offset points')

plt.xlabel(r'Factor de Distancia Cosmológica $K(z)$')
plt.ylabel(r'Retraso Normalizado $\tau_n = \Delta t / E$ (s/GeV)')
plt.title('Cazando Violación de Lorentz (HEASARC Explicit)\nDatos: REDSHIFT, HIGHEST_ENERGY, HIGHEST_ENERGY_TIME', color='white')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n[RESULT] Pendiente LIV: m = {m_fit:.4e} +/- {m_err:.4e} s/GeV")


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



[SYSTEM] Iniciando Protocolo FINAL (Explicit Fields Request)...
[QUERY] Solicitando columnas blindadas: NAME,REDSHIFT,HIGHEST_ENERGY,HIGHEST_ENERGY_TIME
[CRITICAL] Error de conexión HEASARC: Unable to find coordinates for name '%' using https://cds.unistra.fr/cgi-bin/nph-sesame/SNV?%25
Traceback (most recent call last):
  File "/tmp/ipython-input-869082939.py", line 45, in <cell line: 0>
    table = heasarc.query_object(object_name='%', mission=TARGET_TABLE,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/utils/decorators.py", line 143, in deprecated_func
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astroquery/heasarc/core.py", line 463, in query_object
    pos = coordinates.SkyCoord.from_name(object_name)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/astropy/coordinates/sky_coordinate

TypeError: object of type 'NoneType' has no len()